# IndexCalc — Phase 4 테스트

Metric raise/lower, absorb_metric, expand_metric.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from indexcalc import (
    IndexSpace, Tensor, IndexRegistry, parse,
    MetricRegistry, raise_index, lower_index, absorb_metric, expand_metric,
    summarize,
)

In [2]:
# Setup
spacetime = IndexSpace("spacetime", dim=4, indices="μνλρσ", metric="g")
lorentz   = IndexSpace("lorentz",   dim=4, indices="abcde", metric="η")

reg = IndexRegistry()
reg.register(spacetime)
reg.register(lorentz)

# Metric 정의 & 등록
g     = Tensor("g", [spacetime.lower("μ"), spacetime.lower("ν")])
g_inv = Tensor("g", [spacetime.upper("μ"), spacetime.upper("ν")])
eta     = Tensor("η", [lorentz.lower("a"), lorentz.lower("b")])
eta_inv = Tensor("η", [lorentz.upper("a"), lorentz.upper("b")])

metrics = MetricRegistry()
metrics.register(g, g_inv, spacetime)
metrics.register(eta, eta_inv, lorentz)

## 1. raise_index — lower를 upper로 (inverse metric 삽입)

In [3]:
# V_μ → g^{μ,μ_1} V_{μ_1}
V = parse("V_{μ}", reg)
raised = raise_index(V, "μ", metrics)
print(f"V_μ  →  {raised}")
print(f"free: {raised.free_indices}")

V_μ  →  (g^μ^μ_1 * V_μ_1)  [contracted: μ_1]
free: [^μ]


In [4]:
# T^μ_ν 에서 ν를 올리기
T = parse("T^{μ}_{ν}", reg)
T_raised = raise_index(T, "ν", metrics)
print(f"T^μ_ν  →  {T_raised}")
print(f"free: {T_raised.free_indices}")

T^μ_ν  →  (g^ν^μ_1 * T^μ_μ_1)  [contracted: μ_1]
free: [^ν, ^μ]


## 2. lower_index — upper를 lower로 (metric 삽입)

In [5]:
V_up = parse("V^{μ}", reg)
lowered = lower_index(V_up, "μ", metrics)
print(f"V^μ  →  {lowered}")
print(f"free: {lowered.free_indices}")

V^μ  →  (g_μ_μ_1 * V^μ_1)  [contracted: μ_1]
free: [_μ]


## 3. absorb_metric — metric을 텐서에 흡수

In [6]:
# g_{μν} V^{ν} → V_{μ}
expr = parse("g_{μν} V^{ν}", reg)
result = absorb_metric(expr, metrics)
print(f"g_μν V^ν  →  {result}")
print(f"free: {result.free_indices}")

g_μν V^ν  →  V_μ
free: [_μ]


In [7]:
# g^{μν} T_{νλ} → T^{μ}_{λ}
expr = parse("g^{μν} T_{νλ}", reg)
result = absorb_metric(expr, metrics)
print(f"g^μν T_νλ  →  {result}")
print(f"free: {result.free_indices}")

g^μν T_νλ  →  T^μ_λ
free: [^μ, _λ]


In [8]:
# Lorentz metric: η_{ab} V^{b} → V_{a}
expr = parse("η_{ab} V^{b}", reg)
result = absorb_metric(expr, metrics)
print(f"η_ab V^b  →  {result}")
print(f"free: {result.free_indices}")

η_ab V^b  →  V_a
free: [_a]


## 4. expand_metric — absorb의 역연산

In [9]:
# V_μ → g^{μ,μ_1} V_{μ_1}
V_low = parse("V_{μ}", reg)
expanded = expand_metric(V_low, "μ", metrics)
print(f"V_μ  →  {expanded}")
print(f"free: {expanded.free_indices}")

V_μ  →  (g^μ^μ_1 * V_μ_1)  [contracted: μ_1]
free: [^μ]


In [10]:
# V^μ → g_{μ,μ_1} V^{μ_1}
V_up = parse("V^{μ}", reg)
expanded = expand_metric(V_up, "μ", metrics)
print(f"V^μ  →  {expanded}")
print(f"free: {expanded.free_indices}")

V^μ  →  (g_μ_μ_1 * V^μ_1)  [contracted: μ_1]
free: [_μ]


## 5. 왕복 테스트: raise → absorb

In [11]:
original = parse("V_{μ}", reg)
print(f"원본:    {original}   free={original.free_indices}")

step1 = raise_index(original, "μ", metrics)
print(f"raise:   {step1}   free={step1.free_indices}")

step2 = absorb_metric(step1, metrics)
print(f"absorb:  {step2}   free={step2.free_indices}")
print()
print("→ raise 후 absorb하면 index가 올라간 텐서만 남음")

원본:    V_μ   free=[_μ]
raise:   (g^μ^μ_1 * V_μ_1)  [contracted: μ_1]   free=[^μ]
absorb:  V^μ   free=[^μ]

→ raise 후 absorb하면 index가 올라간 텐서만 남음


## 6. 에러 케이스

In [12]:
# 이미 upper인 index를 raise 시도
try:
    raise_index(parse("V^{μ}", reg), "μ", metrics)
except ValueError as e:
    print(f"예상된 에러: {e}")

# 이미 lower인 index를 lower 시도
try:
    lower_index(parse("V_{μ}", reg), "μ", metrics)
except ValueError as e:
    print(f"예상된 에러: {e}")

# 존재하지 않는 index
try:
    raise_index(parse("V_{μ}", reg), "ν", metrics)
except ValueError as e:
    print(f"예상된 에러: {e}")

예상된 에러: Index 'μ' is already upper, cannot raise
예상된 에러: Index 'μ' is already lower, cannot lower
예상된 에러: Index 'ν' not found in free indices: [_μ]
